# Adaptive Energy Brain — Part 5.9: Colab GPU Training

This notebook runs the repository's existing Part 5.8 GNN-Transformer trainer on a Google Colab GPU. It does not duplicate the model, preprocessing, or training loop. It uses train and validation partitions only; the held-out test partition is not loaded. The available dataset is synthetic and hypothetical, not measured campus data.

**Before running:** select **Runtime → Change runtime type → T4 GPU** (or another GPU), configure the repository below, and copy the complete split bundle to Google Drive. Full training is an explicit, potentially long-running cell.

## 1. Introduction and prerequisites

The expected contract is `X [batch, 24, 8, 1]`, `y [batch, 24, 8, 1]`, seven valid kW targets (`B01`–`B05`, `PV01`, `W01`) and a masked `BUS01` context node. The graph is the existing assumed eight-node/14-edge topology without measured edge weights. Historical `global_X` weather features remain excluded from the model. The checked-in data is synthetic/hypothetical.

Use the same Git commit, configs, split bundle, and train-fitted preprocessing artifacts for an experiment. Do not generate new splits or alter graph/target definitions in this notebook.

## 2. GPU runtime verification

Inspect the active Colab runtime before installing or replacing PyTorch. This cell intentionally stops if CUDA is unavailable; enable a GPU runtime and rerun it rather than falling back to CPU.

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, reconnect, and rerun. Full training will not fall back to CPU.")
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cuda:0")
tensor = torch.arange(16, dtype=torch.float32, device=device).reshape(4, 4)
tensor_result = tensor @ tensor.T
assert tensor_result.shape == (4, 4) and torch.isfinite(tensor_result).all()
probe = torch.nn.Linear(4, 2).to(device)
probe_optimizer = torch.optim.SGD(probe.parameters(), lr=0.01)
probe_optimizer.zero_grad(set_to_none=True)
probe_loss = probe(tensor).square().mean()
probe_loss.backward()
assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in probe.parameters())
probe_optimizer.step()
print({"cuda_tensor_operation": "passed", "network_backward": "passed", "gpu": torch.cuda.get_device_name(0)})

## 3. GitHub repository setup

Set a repository URL and a branch/tag/ref or full commit SHA. If this repository has not been pushed to GitHub yet, prepare and push it yourself from the local machine first; this notebook never commits or pushes. For a private repository, add a Colab Secret named `GITHUB_TOKEN` and enable it for this notebook. Never put a token in the URL or notebook.

In [ ]:
import base64
import os
from pathlib import Path
import re
import subprocess
import sys
from urllib.parse import urlsplit

REPOSITORY_URL = ""  # Example: https://github.com/OWNER/REPOSITORY.git
REPOSITORY_REF = "main"  # Branch, tag, or other fetchable ref; ignored when COMMIT_SHA is set.
COMMIT_SHA = ""  # Optional full 40- or 64-character commit SHA.
USE_PRIVATE_REPO = False  # True only when GITHUB_TOKEN is stored in Colab Secrets.
REPO_ROOT = Path("/content/adaptive-energy-brain")

if not REPOSITORY_URL:
    raise ValueError("Set REPOSITORY_URL to the GitHub remote before running this cell. Push the repository manually first if needed.")
parsed_url = urlsplit(REPOSITORY_URL)
if parsed_url.scheme != "https" or not parsed_url.netloc or parsed_url.username or parsed_url.password:
    raise ValueError("Use an HTTPS GitHub URL without embedded credentials.")
if COMMIT_SHA and not re.fullmatch(r"[0-9a-fA-F]{40}|[0-9a-fA-F]{64}", COMMIT_SHA):
    raise ValueError("COMMIT_SHA must be a full 40- or 64-character hexadecimal hash.")

git_env = os.environ.copy()
if USE_PRIVATE_REPO:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Add a nonempty GITHUB_TOKEN in Colab Secrets and enable notebook access.")
    auth = base64.b64encode(f"x-access-token:{github_token}".encode("utf-8")).decode("ascii")
    git_env["GIT_CONFIG_COUNT"] = "1"
    git_env["GIT_CONFIG_KEY_0"] = "http.extraheader"
    git_env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"
    git_env["GIT_TERMINAL_PROMPT"] = "0"
    del github_token, auth

if REPO_ROOT.exists() and not (REPO_ROOT / ".git").exists():
    raise FileExistsError(f"Repository destination exists but is not a Git checkout: {REPO_ROOT}")
if not REPO_ROOT.exists():
    clone = subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPO_ROOT)], env=git_env, text=True, capture_output=True)
    if clone.returncode:
        raise RuntimeError("Repository clone failed. Check the URL, ref, and private-repository Secret. " + clone.stderr[-2000:])
else:
    existing_remote = subprocess.check_output(["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], text=True).strip()
    if existing_remote != REPOSITORY_URL:
        raise RuntimeError("Existing repository checkout has a different origin; use a fresh Colab runtime or the matching URL.")
requested_ref = COMMIT_SHA or REPOSITORY_REF
fetch = subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", requested_ref], env=git_env, text=True, capture_output=True)
if fetch.returncode:
    raise RuntimeError("Could not fetch the selected branch/ref/commit. " + fetch.stderr[-2000:])
checkout = subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"], text=True, capture_output=True)
if checkout.returncode:
    raise RuntimeError("Could not check out the requested repository revision. Ensure no local source edits are present. " + checkout.stderr[-2000:])

ACTUAL_COMMIT_SHA = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
if COMMIT_SHA and ACTUAL_COMMIT_SHA.lower() != COMMIT_SHA.lower():
    raise RuntimeError(f"Checked out {ACTUAL_COMMIT_SHA}, not requested commit {COMMIT_SHA}.")
required_files = ["AGENTS.md", "configs/forecasting_model.yaml", "configs/forecasting_training.yaml", "configs/forecasting_data.yaml", "scripts/train_forecaster.py", "src/forecasting/training.py", "src/forecasting/models/gnn_transformer.py"]
missing_files = [name for name in required_files if not (REPO_ROOT / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Repository revision is missing required project files: {missing_files}")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPOSITORY_URL)
print("Checked-out commit:", ACTUAL_COMMIT_SHA)
print("Working directory:", Path.cwd())

## 4. Dependency installation

The active Colab PyTorch/CUDA build was inspected above. Keep it when CUDA operations succeed; the notebook does **not** reinstall PyTorch or blindly install the optional `requirements-ml-colab.txt` wheel. The project base requirements do not pin/replace PyTorch.

In [ ]:
import subprocess
import sys
import torch

torch_before_install = torch.__version__
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
import importlib
importlib.invalidate_caches()
assert torch.__version__ == torch_before_install, "Base requirements unexpectedly changed PyTorch. Restart and inspect the runtime before proceeding."
if not torch.cuda.is_available():
    raise RuntimeError("CUDA became unavailable after installing base requirements. Re-enable the Colab GPU runtime; do not train on CPU.")
print("Using the existing CUDA-enabled PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

## 5. Google Drive mounting

All experiment output is written directly to Drive so the best and latest checkpoints persist after each completed epoch. Set the project directory below to a generic folder name you choose; no personal Drive path is assumed.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / "adaptive-energy-brain"  # Change to your chosen Drive folder.
DRIVE_DATA_ROOT = DRIVE_PROJECT_DIR / "data/processed"
SPLIT_ROOT = DRIVE_DATA_ROOT / "splits_hypothetical_campus_v1"
PREPROCESSING_PATH = SPLIT_ROOT / "preprocessing.json"
SPLIT_MANIFEST_PATH = SPLIT_ROOT / "split_manifest.json"
TRAINING_OUTPUT_ROOT = DRIVE_PROJECT_DIR / "results/forecasting"
MODEL_CONFIG_PATH = REPO_ROOT / "configs/forecasting_model.yaml"
TRAINING_CONFIG_PATH = REPO_ROOT / "configs/forecasting_training.yaml"
GRAPH_CONFIG_PATH = REPO_ROOT / "configs/forecasting_data.yaml"
TRAINING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Split bundle:", SPLIT_ROOT)
print("Train-fitted preprocessing:", PREPROCESSING_PATH)
print("Graph/sequence metadata:", GRAPH_CONFIG_PATH)
print("Persistent experiment output:", TRAINING_OUTPUT_ROOT)

## 6. Dataset and preprocessing artifact setup

Copy the complete existing split bundle into the Drive location above, preserving its `train/`, `validation/`, `test/`, `split_manifest.json`, and `preprocessing.json` structure. The trainer opens only train and validation records. The test directory is intentionally neither opened nor loaded.

In [ ]:
import json

required_paths = [SPLIT_ROOT / "train", SPLIT_ROOT / "validation", SPLIT_ROOT / "test", PREPROCESSING_PATH, SPLIT_MANIFEST_PATH, MODEL_CONFIG_PATH, TRAINING_CONFIG_PATH, GRAPH_CONFIG_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Required training inputs are missing. Copy the complete split bundle to Drive and check config paths:\n" + "\n".join(missing_paths))
preprocessing_artifact = json.loads(PREPROCESSING_PATH.read_text(encoding="utf-8"))
split_manifest = json.loads(SPLIT_MANIFEST_PATH.read_text(encoding="utf-8"))
if preprocessing_artifact.get("fitted_on") != "train":
    raise ValueError("Preprocessing artifact was not fitted on train; refusing to train.")
if not isinstance(split_manifest.get("checksums"), dict) or not split_manifest["checksums"]:
    raise ValueError("Split manifest has no recorded partition checksums.")
test_manifest_entries = [path for path in split_manifest["checksums"] if path.replace("\\", "/").startswith("test/")]
if not test_manifest_entries:
    raise ValueError("Split manifest has no checksums for the required held-out test partition.")
print("Train, validation, and test directories plus preprocessing and split manifest are present.")
print("Test partition checksummed but not loaded:", len(test_manifest_entries))
print("Scaler fit partition:", preprocessing_artifact["fitted_on"])
print("Manifest checksum entries (metadata only):", len(split_manifest["checksums"]))

## 7. Configuration and dataset verification

The existing Part 5.4 loader verifies train/validation records and uses the saved train-fitted scalers. It does not load test records. This preflight checks chronological boundaries, tensor dimensions, target order/mask, units, graph identity, and the absence of future covariates before any training.

In [ ]:
import yaml
from src.forecasting.dataloader import load_forecasting_train_validation_dataloaders
from src.forecasting.models import GNNTransformerForecaster
from src.forecasting.training import graph_topology_fingerprint

model_config = yaml.safe_load(MODEL_CONFIG_PATH.read_text(encoding="utf-8"))
training_config_yaml = yaml.safe_load(TRAINING_CONFIG_PATH.read_text(encoding="utf-8"))
graph_config = yaml.safe_load(GRAPH_CONFIG_PATH.read_text(encoding="utf-8"))
if model_config["data"]["lookback_hours"] != 24 or model_config["data"]["horizon_hours"] != 24:
    raise ValueError("This Part 5.9 run expects the audited 24-hour lookback and horizon.")
if model_config["targets"]["unit"] != "kW" or model_config["targets"]["nodes"] != ["B01", "B02", "B03", "B04", "B05", "PV01", "W01"]:
    raise ValueError("Target units or target order differ from the audited Part 5 contract.")
if model_config["features"].get("known_future_covariates") != [] or model_config["features"].get("future_realized_observations_as_inputs") is not False:
    raise ValueError("Future covariate policy differs from the audited configuration.")
if graph_config["lookback_hours"] != 24 or graph_config["horizon_hours"] != 24:
    raise ValueError("Sequence configuration and model configuration disagree on window lengths.")

from src.forecasting.training import TrainingConfig
training_config = TrainingConfig.from_mapping(training_config_yaml)
loaders = load_forecasting_train_validation_dataloaders(
    SPLIT_ROOT,
    batch_size=training_config.batch_size,
    seed=training_config.seed,
    device="cuda",
    sequence_config=graph_config,
)
model = GNNTransformerForecaster.from_config(model_config)
if loaders.node_order != tuple(model.graph_node_ids):
    raise ValueError("Dataset/graph node order does not match the model.")
if loaders.node_order != tuple(graph_config["graph"]["nodes"][i]["id"] for i in range(len(graph_config["graph"]["nodes"]))):
    raise ValueError("Loaded graph node order does not match graph configuration.")
if len(loaders.node_order) != 8 or loaders.graph.edge_index.shape[1] != 14 or loaders.graph.edge_weight is not None:
    raise ValueError("Loaded graph differs from the audited unweighted eight-node/14-edge topology.")
if model.target_node_ids != tuple(model_config["targets"]["nodes"]):
    raise ValueError("Model target ordering differs from the configuration.")

train_bounds = loaders.train.dataset.metadata["boundaries"]
validation_bounds = loaders.validation.dataset.metadata["boundaries"]
if train_bounds["end_exclusive_utc"] != validation_bounds["start_utc"]:
    raise ValueError("Train and validation boundaries are not adjacent chronological partitions.")
if loaders.train.dataset.metadata["partition"] != "train" or loaders.validation.dataset.metadata["partition"] != "validation":
    raise ValueError("Only the designated train and validation datasets may be used.")
if loaders.train.dataset.metadata["target_unit"] != "kW" or loaders.validation.dataset.metadata["target_unit"] != "kW":
    raise ValueError("Dataset target units are not kW.")

for dataset in (loaders.train.dataset, loaders.validation.dataset):
    sample = dataset[0]
    expected_valid = set(model.target_node_ids)
    actual_valid = {node for node, valid in zip(loaders.node_order, sample["target_mask"][:, 0].tolist()) if valid}
    if actual_valid != expected_valid or sample["target_mask"][loaders.node_order.index("BUS01")].any():
        raise ValueError(f"Invalid target mask for {dataset.partition}; BUS01 must remain masked.")
    if sample["X"].shape != (24, 8, 1) or sample["y"].shape != (24, 8, 1):
        raise ValueError(f"Unexpected {dataset.partition} tensor dimensions: X={sample['X'].shape}, y={sample['y'].shape}")
    if not sample["input_timestamps"][-1] < sample["target_timestamps"][0]:
        raise ValueError(f"Historical inputs do not precede targets in {dataset.partition}.")

print("train windows:", len(loaders.train.dataset), "validation windows:", len(loaders.validation.dataset))
print("node order:", loaders.node_order, "targets:", model.target_node_ids, "target unit: kW")
print("train boundary:", train_bounds, "validation boundary:", validation_bounds)
print("graph edges:", loaders.graph.edge_index.shape[1], "edge weights:", loaders.graph.edge_weight)
print("graph topology SHA-256:", graph_topology_fingerprint(loaders.graph))
print("Weather global_X is loaded by the dataset but deliberately not passed to this model.")

## 8. GPU smoke test

This runs one actual-data forward/backward pass and optimizer update on CUDA before the full training cell. It verifies the masked loss and finite gradients without using a test batch.

In [ ]:
from src.forecasting.dataloader import move_batch_to_device
from src.forecasting.training import masked_mse_loss

device = torch.device("cuda:0")
model = GNNTransformerForecaster.from_config(model_config).to(device)
model.train()
graph = loaders.graph.to(device)
batch = move_batch_to_device(next(iter(loaders.train)), device)
smoke_optimizer = torch.optim.AdamW(model.parameters(), lr=training_config.learning_rate, weight_decay=training_config.weight_decay)
smoke_optimizer.zero_grad(set_to_none=True)
predictions = model(batch["X"], graph.edge_index, graph.edge_weight, target_mask=batch["target_mask"], node_order=loaders.node_order)
if predictions.shape != batch["y"].shape:
    raise ValueError(f"Prediction/target shape mismatch: {predictions.shape} vs {batch['y'].shape}")
if batch["target_mask"][:, loaders.node_order.index("BUS01"), :].any():
    raise ValueError("BUS01 is unexpectedly marked as a valid target.")
smoke_loss = masked_mse_loss(predictions, batch["y"], batch["target_mask"])
smoke_loss.backward()
if not torch.isfinite(smoke_loss):
    raise FloatingPointError("GPU smoke loss is non-finite.")
missing_gradients = [name for name, parameter in model.named_parameters() if parameter.grad is None]
if missing_gradients or any(not torch.isfinite(parameter.grad).all() for parameter in model.parameters()):
    raise FloatingPointError(f"Missing or non-finite model gradients: {missing_gradients}")
smoke_optimizer.step()
torch.cuda.synchronize()
print({"device": torch.cuda.get_device_name(0), "input_shape": list(batch["X"].shape), "prediction_shape": list(predictions.shape), "masked_mse_kw2": float(smoke_loss.detach().cpu()), "finite_gradients": True, "bus01_included_in_loss": False, "max_memory_mb": round(torch.cuda.max_memory_allocated() / 1024**2, 1)})

## 9. Full training

Review the training YAML and choose a unique experiment ID. The default run is 50 epochs, batch size 32, AdamW, masked MSE, and validation early stopping. Checkpoints are written to Drive after every epoch. Set `RESUME_FROM` to the experiment's `last.pt` after a runtime interruption; keep the same configs and experiment ID. This cell always requires CUDA and never uses the held-out test split.

In [ ]:
from datetime import datetime, timezone
import re
import subprocess
import sys

RESUME_FROM = None  # After interruption, set this to the existing Drive experiment's last.pt.
EXPERIMENT_ID = (Path(RESUME_FROM).parent.name if RESUME_FROM is not None else "part5_9-colab-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
EXPERIMENT_DIR = TRAINING_OUTPUT_ROOT / EXPERIMENT_ID
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", EXPERIMENT_ID):
    raise ValueError("EXPERIMENT_ID contains unsupported characters.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for full training. Enable a Colab GPU runtime.")

command = [sys.executable, str(REPO_ROOT / "scripts/train_forecaster.py"), "--training-config", str(TRAINING_CONFIG_PATH), "--model-config", str(MODEL_CONFIG_PATH), "--sequence-config", str(GRAPH_CONFIG_PATH), "--split-root", str(SPLIT_ROOT), "--output-dir", str(TRAINING_OUTPUT_ROOT), "--experiment-id", EXPERIMENT_ID, "--device", "cuda"]
if RESUME_FROM is not None:
    RESUME_FROM = Path(RESUME_FROM)
    if not RESUME_FROM.is_file():
        raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_FROM}")
    command.extend(["--resume", str(RESUME_FROM)])
elif EXPERIMENT_DIR.exists():
    raise FileExistsError(f"Experiment already exists; choose a fresh ID or explicitly resume: {EXPERIMENT_DIR}")

training_env = os.environ.copy()
for local_storage_key in ("AEB_SPLIT_ROOT", "AEB_EXPERIMENTS_ROOT", "AEB_CHECKPOINTS_ROOT"):
    training_env.pop(local_storage_key, None)
training_env["AEB_GIT_COMMIT_SHA"] = ACTUAL_COMMIT_SHA
print("Running:", " ".join(command))
print("Persistent output:", EXPERIMENT_DIR)
process = subprocess.Popen(command, cwd=REPO_ROOT, env=training_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f"Training CLI exited with status {return_code}. If interrupted, resume from {EXPERIMENT_DIR / 'last.pt'} after reviewing the error.")
print("Training CLI completed. Test data was not used.")

## 10. Training history visualization

Plot the actual epoch-level masked MSE values saved by the training CLI. No held-out test metrics are computed.

In [ ]:
import json
import matplotlib.pyplot as plt

HISTORY_PATH = EXPERIMENT_DIR / "training_history.json"
history_payload = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
if history_payload.get("test_partition_used") is not False:
    raise ValueError("History does not affirm the test partition was unused.")
epochs = [row["epoch"] for row in history_payload["epochs"]]
train_losses = [row["training_loss"] for row in history_payload["epochs"]]
validation_losses = [row["validation_loss"] for row in history_payload["epochs"]]
best_epoch = history_payload["best_epoch"]
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, train_losses, marker="o", label="Train masked MSE")
ax.plot(epochs, validation_losses, marker="o", label="Validation masked MSE")
ax.axvline(best_epoch, color="black", linestyle="--", alpha=0.6, label=f"Best validation epoch: {best_epoch}")
ax.set(xlabel="Epoch", ylabel="Masked MSE (kW²)", title="GNN-Transformer training history")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
LOSS_PLOT_PATH = EXPERIMENT_DIR / "training_history.png"
fig.savefig(LOSS_PLOT_PATH, dpi=160)
plt.show()
print("Saved:", LOSS_PLOT_PATH)

## 11. Checkpoint persistence

Training output is written directly to Google Drive under `results/forecasting/<experiment-id>/`, rather than kept only on Colab's temporary disk. Verify the best checkpoint, latest resumable checkpoint, history, metadata, and plot exist before disconnecting.

In [ ]:
import hashlib
import json

ARTIFACT_PATHS = [EXPERIMENT_DIR / name for name in ("best.pt", "last.pt", "training_history.json", "experiment.json", "training_history.png")]
missing_artifacts = [str(path) for path in ARTIFACT_PATHS if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError("Expected persistent training artifacts are missing: " + ", ".join(missing_artifacts))
metadata = json.loads((EXPERIMENT_DIR / "experiment.json").read_text(encoding="utf-8"))
if metadata["software"].get("source_revision") != ACTUAL_COMMIT_SHA:
    raise ValueError("Experiment metadata does not contain the checked-out Git commit SHA.")
if metadata["software"].get("hardware", {}).get("cuda_available") is not True:
    raise ValueError("Experiment metadata does not confirm CUDA availability.")
for path in ARTIFACT_PATHS[:4]:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(path.name, "bytes=", path.stat().st_size, "sha256=", digest)
print("metadata GPU:", metadata["software"]["hardware"])
print("metadata best epoch/loss:", metadata["training"]["best_epoch"], metadata["training"]["best_validation_loss"])

## 12. Checkpoint reload verification

Reconstruct the model using the saved best checkpoint, the matching graph, and the same Part 2 preprocessing artifacts. Run inference on a validation batch only. The checkpoint does not embed graph edges or scalers, so retain this experiment's split bundle and configs.

In [ ]:
from src.forecasting.dataloader import move_batch_to_device
from src.forecasting.training import load_forecaster_for_evaluation

best_checkpoint_path = EXPERIMENT_DIR / "best.pt"
reloaded_model = load_forecaster_for_evaluation(best_checkpoint_path, device="cuda:0")
validation_batch = move_batch_to_device(next(iter(loaders.validation)), "cuda:0")
validation_graph = loaders.graph.to("cuda:0")
with torch.inference_mode():
    validation_predictions = reloaded_model(validation_batch["X"], validation_graph.edge_index, validation_graph.edge_weight, target_mask=validation_batch["target_mask"], node_order=loaders.node_order)
if validation_predictions.shape != validation_batch["y"].shape or not torch.isfinite(validation_predictions).all():
    raise ValueError("Reloaded checkpoint produced invalid validation predictions.")
bus_index = loaders.node_order.index("BUS01")
if torch.count_nonzero(validation_predictions[:, :, bus_index, :]).item() != 0:
    raise ValueError("Reloaded model did not preserve the BUS01 masked placeholder.")
if tuple(reloaded_model.target_node_ids) != tuple(model_config["targets"]["nodes"]):
    raise ValueError("Reloaded checkpoint target ordering differs from the configured order.")
print({"reload": "passed", "validation_shape": list(validation_predictions.shape), "finite": True, "target_order": reloaded_model.target_node_ids, "BUS01_zero_placeholder": True, "test_partition_used": False})

## 13. Results and next steps

Keep the experiment directory, `experiment.json`, `training_history.json`, loss plot, best/latest checkpoints, split manifest, preprocessing artifact, and exact Git revision together. Full GPU training is complete only after the preceding cells ran successfully on a GPU and the reload checks passed. Do not evaluate the held-out test partition here. Part 5.10 should begin only after reviewing the saved run and this evidence.

In [ ]:
print("Experiment:", EXPERIMENT_ID)
print("Commit:", ACTUAL_COMMIT_SHA)
print("GPU:", torch.cuda.get_device_name(0))
print("Best validation epoch:", history_payload["best_epoch"])
print("Best validation masked MSE (kW²):", history_payload["best_validation_loss"])
print("Duration seconds:", history_payload["duration_seconds"])
print("Persistent artifacts:", EXPERIMENT_DIR)
print("Test split used:", False)
print("Part 5.10 readiness: requires manual review of this completed Colab run; no final test evaluation was performed.")